# Desafio Final — Análise de Dados de Energia com API Pública

**Curso:** Ciência da Computação  
**Disciplina:** Soluções em Energias Renováveis e Sustentáveis

## Situação-problema

Uma equipe de planejamento energético precisa analisar o comportamento da carga elétrica de uma região atendida pelo Sistema Interligado Nacional (SIN).

Os dados serão obtidos diretamente de uma API pública do **Operador Nacional do Sistema Elétrico (ONS)**. A conexão com a API e a preparação inicial do JSON já estão fornecidas. A partir daí, sua equipe deverá construir o DataFrame, organizar os dados, criar recortes, calcular indicadores, produzir gráficos e elaborar um relatório técnico.

> Todos os códigos, resultados, gráficos e respostas devem permanecer neste mesmo Notebook.

## 1. Fonte dos dados

API pública de **Carga Verificada do ONS**:

- Portal: https://dados.ons.org.br/
- Conjunto de dados: https://dados.ons.org.br/dataset/carga-energia-verificada

Neste notebook será utilizada inicialmente a área **SP — São Paulo**, no período de **01/08/2025 a 07/08/2025**.

## 2. Bibliotecas

In [1]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'requests'

## 3. Consulta à API

Esta célula está pronta. Não é necessário conhecer `requests` para realizar o desafio.

In [ ]:
url = "https://apicarga.ons.org.br/prd/cargaverificada"

parametros = {
    "dat_inicio": "2025-08-01",
    "dat_fim": "2025-08-07",
    "cod_areacarga": "SP"
}

response = requests.get(url, params=parametros, timeout=30)

print("Status:", response.status_code)
print("URL:", response.url)

response.raise_for_status()
dados_json = response.json()

## 4. Preparação inicial do JSON

A célula abaixo localiza a lista principal de registros retornada pela API e a armazena em `registros`.

In [ ]:
if isinstance(dados_json, list):
    registros = dados_json

elif isinstance(dados_json, dict):
    chaves_com_lista = [
        chave for chave, valor in dados_json.items()
        if isinstance(valor, list)
    ]

    if not chaves_com_lista:
        raise ValueError("A resposta não contém uma lista de registros.")

    chave_registros = chaves_com_lista[0]
    registros = dados_json[chave_registros]
    print("Chave utilizada:", chave_registros)

else:
    raise TypeError("Formato de JSON não reconhecido.")

print("Tipo:", type(registros))
print("Quantidade de registros:", len(registros))

if registros:
    print("\nPrimeiro registro:")
    print(registros[0])

In [ ]:
type(registros)

In [ ]:
registros

# A partir daqui, o trabalho é da equipe

## DESAFIO 1 — Construção e inspeção do DataFrame

1. Crie um DataFrame Pandas chamado `dados` a partir de `registros`.
2. Exiba os primeiros registros.
3. Determine a quantidade de linhas e colunas.
4. Liste os nomes dos atributos.
5. Utilize `info()`.
6. Utilize `describe()`.
7. Em Markdown, identifique quais atributos representam data/hora, área de carga e valor de carga.

**Antes de prosseguir, compreenda a estrutura efetivamente retornada pela API.**

In [ ]:
# DESAFIO 1 — Construção e inspeção do DataFrame

dados = pd.DataFrame(registros)

print("Primeiras linhas do DataFrame:")
display(dados.head())

n_linhas, n_colunas = dados.shape
print(f"\nQuantidade de linhas: {n_linhas}")
print(f"Quantidade de colunas: {n_colunas}")

print("\nNomes dos atributos (colunas):")
print(list(dados.columns))

print("\nInformações gerais (info):")
dados.info()

print("\nEstatísticas descritivas (describe):")
display(dados.describe(include="all"))


In [ ]:

from IPython.display import display, Markdown

col_datahora = None
col_area = None
col_valor = None

for col in dados.columns:
    nome = col.lower()
    if col_datahora is None and any(p in nome for p in ["din", "data", "hora", "date", "time"]):
        col_datahora = col
    if col_area is None and any(p in nome for p in ["area", "subsistema", "estado", "regiao"]):
        col_area = col
    if (
        col_valor is None
        and any(p in nome for p in ["val", "carga", "mwmed", "mw"])
        and pd.api.types.is_numeric_dtype(dados[col])
    ):
        col_valor = col


if col_valor is None:
    colunas_numericas = dados.select_dtypes(include="number").columns
    if len(colunas_numericas) > 0:
        col_valor = dados[colunas_numericas].var().idxmax()

display(Markdown(f"""


- **Data/hora:** `{col_datahora}`
- **Área de carga:** `{col_area}`
- **Valor de carga:** `{col_valor}`

*Identificação feita a partir dos nomes e tipos das colunas retornadas pela API. Confirme comparando com `dados.head()` e `dados.info()` acima — se a API tiver mudado nomes de campo, ajuste as variáveis `col_datahora`, `col_area` e `col_valor` manualmente aqui.*
"""))


## DESAFIO 2 — Organização dos dados

1. Renomeie os principais atributos com nomes mais simples.
2. Crie um novo DataFrame contendo apenas os atributos necessários.
3. Verifique valores ausentes.
4. Caso existam, informe quantos há em cada atributo relevante.
5. Verifique se a variável de carga está em formato numérico.
6. Verifique como a data/hora está representada.
7. Registre em Markdown qualquer decisão de tratamento.

In [ ]:
# DESAFIO 2 — Organização dos dados

mapa_nomes = {}
if col_datahora:
    mapa_nomes[col_datahora] = "data_hora"
if col_area:
    mapa_nomes[col_area] = "area_carga"
if col_valor:
    mapa_nomes[col_valor] = "carga"

dados_renomeado = dados.rename(columns=mapa_nomes)

colunas_uteis = [v for v in mapa_nomes.values() if v in dados_renomeado.columns]
dados_organizados = dados_renomeado[colunas_uteis].copy()

print("Colunas selecionadas:", colunas_uteis)
display(dados_organizados.head())

print("\nValores ausentes por atributo:")
print(dados_organizados.isna().sum())

print("\nTipo de dado de 'carga' antes do tratamento:", dados_organizados["carga"].dtype)
dados_organizados["carga"] = pd.to_numeric(dados_organizados["carga"], errors="coerce")
print("Tipo de dado de 'carga' depois do tratamento:", dados_organizados["carga"].dtype)

print("\nExemplo de valores de 'data_hora' antes do tratamento:")
print(dados_organizados["data_hora"].head())
dados_organizados["data_hora"] = pd.to_datetime(dados_organizados["data_hora"], errors="coerce")
print("\nTipo de 'data_hora' após conversão:", dados_organizados["data_hora"].dtype)

qtd_antes = len(dados_organizados)
dados_organizados = dados_organizados.dropna(subset=["carga", "data_hora"]).sort_values("data_hora").reset_index(drop=True)
qtd_depois = len(dados_organizados)

print(f"\nRegistros antes da limpeza: {qtd_antes} | após remover ausentes/inválidos: {qtd_depois}")

display(Markdown(f"""
- Registros sem `carga` ou `data_hora` válidos foram descartados: de {qtd_antes} para {qtd_depois} registros.
"""))


## DESAFIO 3 — Indicadores da carga elétrica

Calcule:

1. carga mínima;
2. carga máxima;
3. carga média;
4. mediana;
5. amplitude entre máximo e mínimo;
6. quantidade total de medições.

Depois responda:

**O valor máximo está muito distante do comportamento médio observado? Justifique com os indicadores calculados.**

In [ ]:
# DESAFIO 3 — Indicadores da carga elétrica

carga_min = dados_organizados["carga"].min()
carga_max = dados_organizados["carga"].max()
carga_media = dados_organizados["carga"].mean()
carga_mediana = dados_organizados["carga"].median()
amplitude = carga_max - carga_min
total_medicoes = dados_organizados["carga"].count()

print(f"Carga mínima: {carga_min:.2f}")
print(f"Carga máxima: {carga_max:.2f}")
print(f"Carga média: {carga_media:.2f}")
print(f"Mediana: {carga_mediana:.2f}")
print(f"Amplitude (máximo - mínimo): {amplitude:.2f}")
print(f"Quantidade total de medições: {total_medicoes}")

distancia_pct = (carga_max - carga_media) / carga_media * 100 if carga_media else float("nan")

display(Markdown(f"""

O valor máximo ({carga_max:.2f}) está {distancia_pct:.1f}% acima da carga média ({carga_media:.2f}).
{"Essa diferença é relativamente pequena, o que sugere que o pico não é um evento muito atípico frente ao comportamento médio observado no período." if distancia_pct < 15 else "Essa diferença é expressiva, o que sugere que o pico representa um evento bem acima do comportamento médio observado no período."}
Comparando média ({carga_media:.2f}) e mediana ({carga_mediana:.2f}): {"os valores são próximos, indicando distribuição pouco assimétrica." if abs(carga_media - carga_mediana) / carga_media < 0.05 else "há diferença perceptível entre elas, indicando alguma assimetria na distribuição dos valores de carga."}
"""))


## DESAFIO 4 — Períodos de alta demanda

Considere como **alta demanda** os registros com carga superior a **90% da carga máxima**.

1. Calcule o limiar.
2. Crie um novo DataFrame com os registros acima dele.
3. Conte os registros.
4. Calcule o percentual em relação ao total.
5. Identifique o maior valor de carga.
6. Identifique a data e o horário do pico, quando disponíveis.

Responda:

**Os períodos próximos ao pico representam uma parcela grande ou pequena do período analisado?**

In [ ]:
# DESAFIO 4 — Períodos de alta demanda

limiar_alta_demanda = 0.9 * carga_max
alta_demanda = dados_organizados[dados_organizados["carga"] > limiar_alta_demanda].copy()

qtd_alta_demanda = len(alta_demanda)
pct_alta_demanda = qtd_alta_demanda / total_medicoes * 100

print(f"Limiar de alta demanda (90% da carga máxima): {limiar_alta_demanda:.2f}")
display(alta_demanda.head())
print(f"Quantidade de registros em alta demanda: {qtd_alta_demanda}")
print(f"Percentual em relação ao total: {pct_alta_demanda:.2f}%")

maior_valor_alta_demanda = alta_demanda["carga"].max() if qtd_alta_demanda > 0 else None
if qtd_alta_demanda > 0:
    momento_pico = alta_demanda.loc[alta_demanda["carga"].idxmax(), "data_hora"]
else:
    momento_pico = None

print(f"Maior valor de carga: {maior_valor_alta_demanda}")
print(f"Data e horário do pico: {momento_pico}")

display(Markdown(f"""
**Resposta (DESAFIO 4)**

Os registros de alta demanda (carga acima de {limiar_alta_demanda:.2f}) correspondem a {pct_alta_demanda:.2f}% do total de medições do período analisado.
{"Isso representa uma parcela pequena do período, indicando que os momentos próximos ao pico são pontuais, concentrados em poucos horários." if pct_alta_demanda < 20 else "Isso representa uma parcela considerável do período, indicando que a carga permanece próxima do valor de pico por boa parte do tempo analisado."}
"""))


## DESAFIO 5 — Segundo critério de análise

Crie um segundo recorte dos dados utilizando um critério definido pela equipe.

Possibilidades:

- carga acima da média;
- carga abaixo de uma porcentagem do máximo;
- intervalo de valores;
- determinado dia ou período;
- combinação de duas condições.

Apresente:

1. o critério escolhido;
2. o novo DataFrame;
3. a quantidade de registros;
4. o percentual;
5. a comparação com o conjunto de alta demanda.

In [ ]:
# DESAFIO 5 — Segundo critério de análise
# Critério escolhido pela equipe: registros com carga ACIMA DA MÉDIA do período

criterio_2_descricao = "Carga acima da média do período"

acima_da_media = dados_organizados[dados_organizados["carga"] > carga_media].copy()

qtd_acima_media = len(acima_da_media)
pct_acima_media = qtd_acima_media / total_medicoes * 100

print("Critério escolhido:", criterio_2_descricao)
display(acima_da_media.head())
print(f"Quantidade de registros: {qtd_acima_media}")
print(f"Percentual em relação ao total: {pct_acima_media:.2f}%")

display(Markdown(f"""


- Alta demanda (carga > 90% do máximo): {qtd_alta_demanda} registros ({pct_alta_demanda:.2f}%)
- Acima da média: {qtd_acima_media} registros ({pct_acima_media:.2f}%)

{"O critério 'acima da média' é bem mais abrangente que o de alta demanda, já que a média é superada com muito mais frequência do que o limiar de 90% do pico." if pct_acima_media > pct_alta_demanda else "Os dois critérios resultaram em parcelas semelhantes do período analisado."}
"""))


## DESAFIO 6 — Visualização

Construa **pelo menos dois gráficos**.

- Um deve representar o comportamento da carga ao longo das observações ou do tempo.
- O segundo deve ser escolhido pela equipe.

Inclua título, eixos e unidades quando aplicável.

Após cada gráfico, escreva uma interpretação curta.

In [ ]:
# DESAFIO 6 — Visualização

# Gráfico 1: comportamento da carga ao longo do tempo
plt.figure(figsize=(12, 5))
plt.plot(dados_organizados["data_hora"], dados_organizados["carga"], color="steelblue", linewidth=1)
plt.axhline(limiar_alta_demanda, color="red", linestyle="--", label="Limiar de alta demanda (90% do máx.)")
plt.title("Carga verificada ao longo do tempo — Área SP (01/08/2025 a 07/08/2025)")
plt.xlabel("Data/Hora")
plt.ylabel("Carga (MWmed)")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
display(Markdown(f"""
**Interpretação do Gráfico 1**

A série mostra a variação da carga verificada ao longo do período analisado, com os picos ultrapassando o limiar de {limiar_alta_demanda:.2f} MWmed destacado em vermelho.
{"Observa-se um padrão cíclico, compatível com variações diárias de consumo (picos e vales se repetindo dia após dia)." if total_medicoes > 24 else "O volume de registros é pequeno para afirmar um padrão cíclico com segurança."}
"""))


In [ ]:
# Gráfico 2: distribuição dos valores de carga
plt.figure(figsize=(8, 5))
sns.histplot(dados_organizados["carga"], bins=30, kde=True, color="seagreen")
plt.axvline(carga_media, color="orange", linestyle="--", label=f"Média ({carga_media:.0f})")
plt.axvline(carga_mediana, color="purple", linestyle="--", label=f"Mediana ({carga_mediana:.0f})")
plt.title("Distribuição dos valores de carga verificada")
plt.xlabel("Carga (MWmed)")
plt.ylabel("Frequência")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
display(Markdown(f"""


O histograma mostra como os valores de carga se distribuem ao longo do período: concentração em torno da média ({carga_media:.2f}) e cauda em direção ao valor máximo ({carga_max:.2f}), consistente com o percentual de {pct_alta_demanda:.2f}% de registros classificados como alta demanda.
"""))


## DESAFIO 7 — Síntese para o relatório

Crie uma variável `resumo_resultados` contendo pelo menos:

- região;
- período;
- quantidade de registros;
- carga mínima;
- carga máxima;
- carga média;
- mediana;
- limiar de alta demanda;
- quantidade e percentual de alta demanda;
- momento do pico;
- resultado do segundo critério.

A IA deverá receber **resultados produzidos por vocês**, e não substituir a análise.

In [ ]:
# DESAFIO 7 — Síntese para o relatório

regioes_analisadas = dados_organizados["area_carga"].unique().tolist() if "area_carga" in dados_organizados.columns else ["SP"]
periodo_inicio = dados_organizados["data_hora"].min()
periodo_fim = dados_organizados["data_hora"].max()

resumo_resultados = f"""
Região analisada: {regioes_analisadas}
Período analisado: {periodo_inicio} a {periodo_fim}
Quantidade de registros: {total_medicoes}
Carga mínima: {carga_min:.2f} MWmed
Carga máxima: {carga_max:.2f} MWmed
Carga média: {carga_media:.2f} MWmed
Mediana: {carga_mediana:.2f} MWmed
Limiar de alta demanda (90% do máximo): {limiar_alta_demanda:.2f} MWmed
Registros de alta demanda: {qtd_alta_demanda} ({pct_alta_demanda:.2f}% do total)
Momento do pico: {momento_pico}
Segundo critério ({criterio_2_descricao}): {qtd_acima_media} registros ({pct_acima_media:.2f}% do total)
"""

print(resumo_resultados)


# OPCIONAL: Relatório técnico com apoio do Gemini

No Colab:

1. abra **Secrets**;
2. crie `GEMINI_API_KEY`;
3. informe sua chave;
4. permita o acesso do notebook ao secret.

**Não coloque a chave diretamente no código.**

In [ ]:
!pip -q install -U google-genai

In [ ]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

## DESAFIO 8 — Geração do relatório

O relatório deve:

- usar os resultados calculados;
- apresentar os principais indicadores;
- destacar o pico e os períodos de alta demanda;
- comparar os dois critérios;
- não inventar causas;
- diferenciar observações de hipóteses;
- terminar com uma conclusão.

In [ ]:
prompt = f'''

{resumo_resultados}
'''

In [ ]:
response_gemini = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt
)

print(response_gemini.text)

## DESAFIO 9 — Validação crítica

Compare o texto gerado pelo Gemini com os cálculos, DataFrames e gráficos.

Responda:

1. Os indicadores foram utilizados corretamente?
2. Há alguma afirmação que não pode ser confirmada pelos dados?
3. Houve interpretação exagerada ou causalidade não demonstrada?
4. Que alterações a equipe realizou no texto?

## Relatório final

Insira abaixo a versão final revisada pela equipe.

_Escreva aqui a versão final revisada._

---

## Entrega

O notebook deve apresentar:

- consulta à API executada;
- DataFrame criado;
- inspeção e organização dos dados;
- indicadores;
- pelo menos dois DataFrames derivados por critérios;
- percentuais;
- pelo menos dois gráficos;
- interpretações;
- síntese para a IA;
- relatório com apoio do Gemini;
- validação crítica;
- versão final revisada.

**Entregue somente este Notebook (.ipynb), com todas as células executadas e os resultados visíveis.**